## Sync vs Async: How can Async help with LLM calls ?

The simplest way to understand async and await is:

* async says:  “this function can pause while waiting.”

* await says: “pause this function here, but let other work run meanwhile.”

So async is particularly useful for I/O-bound work:

- HTTP requests
- calling LLM APIs
- database queries
- reading network streams
- waiting for external services

### 1. Normal synchronous Python

Imagine you need to fetch 3 URLs:

In [ ]:
import time

def fetch(url):
    print(f"Starting {url}")
    time.sleep(2)       # pretend we're waiting for network
    print(f"Finished {url}")


def main():
    start_time = time.perf_counter()

    fetch("google.com")
    fetch("openai.com")
    fetch("github.com")

    end_time = time.perf_counter()

    print(f"\nStart time: {start_time:.2f}")
    print(f"End time:   {end_time:.2f}")
    print(f"Total time: {end_time - start_time:.2f} seconds")
    
main()

Starting google.com
Finished google.com
Starting openai.com
Finished openai.com
Starting github.com
Finished github.com

Start time: 605286.79
End time:   605292.79
Total time: 6.01 seconds


### 2. Now async and await

In [ ]:
import asyncio

async def fetch(url):
    print(f"Starting {url}")

    await asyncio.sleep(2)

    print(f"Finished {url}")
    

Notice two things:

```
async def
async def fetch(url):
```

means:

This function is an async function (coroutine). It can pause at await.

```
await
await asyncio.sleep(2)
```

means:

"I'm waiting for something. Pause me here and allow the event loop to run other async tasks."

### 3. The important part: running multiple tasks

In [8]:
import asyncio
import time

async def fetch(url):
    print(f"Starting {url}")
    
    # Pretend we're making a network request
    await asyncio.sleep(2)
    
    print(f"Finished {url}")


async def main():
    start_time = time.perf_counter()

    await asyncio.gather(
        fetch("google.com"),
        fetch("openai.com"),
        fetch("github.com")
    )

    end_time = time.perf_counter()

    print(f"\nStart time: {start_time:.2f}")
    print(f"End time:   {end_time:.2f}")
    print(f"Total time: {end_time - start_time:.2f} seconds")



# ❌ Don't do this in Jupyter
# asyncio.run(main())
# RuntimeError: asyncio.run() cannot be called from a running event loop


# ✅ Jupyter already has an event loop
await main()

Starting google.com
Starting openai.com
Starting github.com
Finished google.com
Finished openai.com
Finished github.com

Start time: 605419.40
End time:   605421.40
Total time: 2.00 seconds


### The key difference
SYNC:

```
Google  ──────2s──────
                       OpenAI ──────2s──────
                                              GitHub ──────2s──────
                       TOTAL ≈ 6 seconds
```



ASYNC:

```
Google  ──────2s──────
OpenAI  ──────2s──────
GitHub  ──────2s──────
          ↑
       concurrent
```
       
TOTAL ≈ 2 seconds

This is the fundamental reason async is useful for LLM API calls and HTTP requests: you're usually spending most of the time waiting for a response, not using the CPU.

### 4. What actually happened?

Think of the event loop as a manager.

You give it:
```
fetch Google
fetch OpenAI
fetch GitHub
```
It starts Google:

Google: "I'm waiting for network..."

Instead of sitting there doing nothing, it starts OpenAI:

OpenAI: "I'm waiting for network..."

Then GitHub:

GitHub: "I'm waiting for network..."

While all three are waiting, the event loop can handle other tasks.

Eventually:
```
Google → response arrives
OpenAI → response arrives
GitHub → response arrives
```

### 5. Very important: await does NOT mean "run concurrently"

This is a common misunderstanding.

If you do:

In [9]:
async def main():
    await fetch("google.com")
    await fetch("openai.com")
    await fetch("github.com")

you're effectively doing:

```
Google → wait → finish
                 ↓
              OpenAI → wait → finish
                            ↓
                         GitHub → wait → finish
```

That's still sequential.

To get concurrency, you need to create/schedule multiple tasks, for example:

In [10]:
await asyncio.gather(
    fetch("google.com"),
    fetch("openai.com"),
    fetch("github.com")
)

Starting google.com
Starting openai.com
Starting github.com
Finished google.com
Finished openai.com
Finished github.com


[None, None, None]

### 6. Connecting this to AI/LLM engineering example

Suppose you need to call 5 LLM APIs.

Without async:


```
response1 = call_llm(prompt1)  # 3 sec
response2 = call_llm(prompt2)  # 3 sec
response3 = call_llm(prompt3)  # 3 sec
response4 = call_llm(prompt4)  # 3 sec
response5 = call_llm(prompt5)  # 3 sec
```

Potentially:

15 seconds

With async:

```
responses = await asyncio.gather(
    call_llm(prompt1),
    call_llm(prompt2),
    call_llm(prompt3),
    call_llm(prompt4),
    call_llm(prompt5)
)
```

Potentially:

~3 seconds

because the requests are waiting concurrently.

## 7. How does the semaphore come in?

Suppose you have:

1,000 URLs

You don't want to launch 1,000 requests simultaneously.

You want:

10 running
↓
10 running
↓
10 running
↓
...

That's where a semaphore comes in.

```
semaphore = asyncio.Semaphore(10)
```

Then:

```
async def fetch(url):
    async with semaphore:
        # only 10 functions can be inside here
        response = await make_request(url)
```

Think of it as a room with 10 chairs.
```
              SEMAPHORE
          ┌──────────────┐
          │  chair chair │
          │  chair chair │
          │  chair chair │
          │  chair chair │
          │  chair chair │
          └──────────────┘
               10 seats
```
100 URLs want to enter.

Only 10 can enter.

The other 90 wait.

When one finishes:

URL #3 leaves
       ↓
chair becomes free
       ↓
URL #11 enters

So:

- async → allows a function to pause
- await → pauses it while waiting for I/O
- gather() → lets many async operations run concurrently
- Semaphore(10) → limits how many can run at once

That's the core mental model you need before getting into async HTTP requests, retries, exponential backoff, and LLM API calls.